# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AzramAfaq/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**One row = one content page, on one specific day, for one client.** Verified: grouping by (report_date, client_hash_id, content_hash_id) and checking for duplicates returned 0 rows — the grain holds.

**Time window:** using `month=2026-03` (March 2026) as my working/iteration month — 9,841,378 rows, dates confirmed to run cleanly from 2026-03-01 to 2026-03-31. The most recent month (June 2026) is deliberately left untouched, since it's the natural outcome window for any past→future label.

**Availability caveat (verified):** only 36.7% of rows have real Search Console data (`gsc_data_available IS TRUE`), and only 4.2% have real Analytics data (`ga4_data_available IS TRUE`). Most rows in this table exist but carry no usable signal — this shapes every downstream count I report from here on.

In [1]:
!pip install duckdb --quiet

In [2]:
import duckdb

In [3]:
from google.colab import userdata

In [4]:
hf_token = userdata.get('HF_TOKEN')

In [5]:
con = duckdb.connect()

In [6]:

con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

In [8]:
con.sql("SELECT COUNT(*) FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet')")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│          104 │
└──────────────┘

In [9]:
con.sql("DESCRIBE SELECT * FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet')")

┌─────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│     column_name     │ column_type │  null   │   key   │ default │  extra  │
│       varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ client_hash_id      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ is_active           │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ has_gsc_access      │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ has_ga4_access      │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ access_profile      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_created_date │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_updated_date │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_start      │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_start      │ DATE        │ YES     │ NULL    │ NULL  

In [10]:
con.sql("DESCRIBE SELECT * FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')")

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

In [11]:
con.sql("""
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
GROUP BY report_date, client_hash_id, content_hash_id
HAVING c > 1
LIMIT 5
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────────┬─────────────────┬───────┐
│ report_date │ client_hash_id │ content_hash_id │   c   │
│    date     │    varchar     │     varchar     │ int64 │
├─────────────┴────────────────┴─────────────────┴───────┤
│                         0 rows                         │
└────────────────────────────────────────────────────────┘

In [12]:
con.sql("""
SELECT COUNT(*) AS row_count, MIN(report_date) AS earliest_date, MAX(report_date) AS latest_date
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
""")

┌───────────┬───────────────┬─────────────┐
│ row_count │ earliest_date │ latest_date │
│   int64   │     date      │    date     │
├───────────┼───────────────┼─────────────┤
│   9841378 │ 2026-03-01    │ 2026-03-31  │
└───────────┴───────────────┴─────────────┘

In [13]:
con.sql("""
SELECT
  COUNT(*) AS total_rows,
  COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
  COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
""")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────────┬────────────────────┐
│ total_rows │ gsc_available_rows │ ga4_available_rows │
│   int64    │       int64        │       int64        │
├────────────┼────────────────────┼────────────────────┤
│    9841378 │            3611061 │             413966 │
└────────────┴────────────────────┴────────────────────┘

## 2. Fields: feature / label / context / excluded

## 2. Fields: feature / label / context / excluded

**Context** (for filtering/joining/scoping — never fed to a model):
`report_date`, `client_hash_id`, `content_hash_id`, `month` — identify and locate each row.
`client_has_gsc`, `client_has_ga4`, `gsc_data_available`, `ga4_data_available` — these tell me
*whether* a row's data is trustworthy, but they describe data-collection plumbing, not the
content itself — used only to decide what to include, never as a signal the model learns from.

**Label / proxy** (the thing being predicted, or built from — never a feature):
`gsc_clicks`, `gsc_impressions`, `ga4_sessions`, `ga4_pageviews` — these are the outcome metrics.
My eventual target (a page's decline/priority score) will be built by comparing these values
across two separate time windows. Because the target is *built from* these columns, none of them
can also be used as a feature — that would let the model see its own answer.

**Feature** (knowable before the decision moment, safe to use):
`gsc_avg_position` — search ranking position, known day-to-day, independent of the label window.
`sessions_organic`, `sessions_direct`, `sessions_referral`, `sessions_social`, `sessions_paid`,
`sessions_ai` — traffic-channel mix, describes *how* a page gets visited, not the outcome I'm
predicting. `ai_chatgpt`, `ai_perplexity`, `ai_gemini`, `ai_copilot`, `ai_claude`, `ai_meta`,
`ai_other` — breakdown of AI-referral traffic by source. `ga4_engaged_sessions`,
`ga4_total_engagement_sec`, `scroll_events` — on-page engagement quality signals, separate from
raw traffic volume.

**Excluded** (each with a one-line why):
`gsc_sum_position` — excluded because it's a raw sum, not an average; its scale depends entirely
on how many impressions a page had, which makes it a near-duplicate of
`gsc_impressions × gsc_avg_position` rather than genuinely new information — keeping both invites
redundant/confounded features.

In [14]:
cols = con.sql("DESCRIBE SELECT * FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')").df()
print(cols["column_name"].tolist())

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


In [15]:
features = con.sql("""
SELECT
  content_hash_id,
  AVG(gsc_avg_position) AS avg_position,
  SUM(sessions_organic) AS total_organic_sessions,
  SUM(sessions_ai) AS total_ai_sessions,
  SUM(ga4_engaged_sessions) AS total_engaged_sessions,
  SUM(scroll_events) AS total_scroll_events,
  SUM(gsc_clicks) AS total_clicks
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
WHERE gsc_data_available IS TRUE
GROUP BY content_hash_id
""").df()

print(features.shape)
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(176738, 7)


,content_hash_id,avg_position,total_organic_sessions,total_ai_sessions,total_engaged_sessions,total_scroll_events,total_clicks
0,content_7a105f548d9c6916,7.209549,1.0,0.0,0.0,0.0,7.0
1,content_a3ea9792f793ec72,2.987198,0.0,0.0,0.0,0.0,0.0
2,content_36c36abc7650d7af,6.724039,6.0,0.0,0.0,0.0,6.0
3,content_a7da352b73b02668,7.244844,1.0,0.0,0.0,0.0,13.0
4,content_1855a661b4d36130,4.209227,0.0,0.0,0.0,1.0,1.0


In [20]:
X = features[honest_features].fillna(0)
y = features["total_clicks"].fillna(0)

model = LinearRegression().fit(X, y)
honest_score = r2_score(y, model.predict(X))
print(f"Honest score (5 real features only): {honest_score:.3f}")

Honest score (5 real features only): 0.610


## 3. Verify it with queries (grain, counts, missing values, windows)
## 3. Verify it with queries (grain, counts, missing values, windows) + five features + the trap

**Five features built for March 2026, one row per page** (176,738 pages, filtered to `gsc_data_available IS TRUE`):

1. `avg_position` — knowable at the decision moment because it's the page's average search ranking that month, independent of any future outcome.
2. `total_organic_sessions` — knowable because it's historical traffic already observed, not a future event.
3. `total_ai_sessions` — same: observed AI-referral traffic, already happened.
4. `total_engaged_sessions` — an engagement-quality signal from the same historical window, not the outcome window.
5. `total_scroll_events` — on-page behavior already observed, describes how visitors interacted, not whether they'll return.

**Honest baseline score (5 real features predicting `total_clicks`): 0.610** (R²).

**The trap, performed on purpose:** added one label-derived column, `leaky_avg_daily_clicks`, computed directly from `total_clicks` (the same value we're predicting). The score jumped from 0.610 to a suspicious **1.000** — a perfect score, which is the tell that the model wasn't learning a real pattern, it was just being handed the answer. Removed the column immediately after confirming this, and kept only the honest 0.610 as the real result.

In [22]:
features["leaky_avg_daily_clicks"] = features["total_clicks"] / 30

leaky_features = honest_features + ["leaky_avg_daily_clicks"]
X_leaky = features[leaky_features].fillna(0)

model_leaky = LinearRegression().fit(X_leaky, y)
leaky_score = r2_score(y, model_leaky.predict(X_leaky))
print(f"Leaky score (with the cheat column): {leaky_score:.3f}")

Leaky score (with the cheat column): 1.000


In [23]:
del features["leaky_avg_daily_clicks"]
print(f"Final honest score (kept): {honest_score:.3f}")
print("Leaky column removed — only the 5 real features remain.")

Final honest score (kept): 0.610
Leaky column removed — only the 5 real features remain.


## 4. Data limits

## 4. Data limits

**This data cannot tell you what it never measured.** The biggest limitation, verified with a
query earlier: in March 2026, only **36.7%** of rows have real Search Console data
(`gsc_data_available IS TRUE`), and only **4.2%** have real Analytics data
(`ga4_data_available IS TRUE`). This isn't random missingness — it traces back to
`dim_clients.gsc_data_start` / `ga4_data_start`, meaning different clients connected these tools
at different points in time. A page belonging to a client who connected GA4 late will show up as
"no analytics data" for months it was actually live — that's a gap in tracking history, not
evidence the page had zero visitors.

Practically, this means: any feature or label built from `ga4_*` columns is only trustworthy for
the ~4% of rows where that data genuinely exists, and any conclusion I draw from this slice
describes clients/pages with GSC or GA4 already connected — not the full, true population of
all content. I filtered with `IS TRUE` throughout specifically to keep this honest, rather than
silently treating missing data as zero.

In [24]:
# Evidence for this limitation already shown in Section 1's Query 3:
# gsc_available_rows = 3,611,061 of 9,841,378 (36.7%)
# ga4_available_rows =   413,966 of 9,841,378 (4.2%)
print("See Section 1, Query 3 for the availability numbers backing this limitation.")

See Section 1, Query 3 for the availability numbers backing this limitation.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.